# Grid Pulse EV — Step 1: Synthetic EV Sessions & Uncontrolled Baseline

**Research:** Optimizing Public EV Charging Schedules for Renewable-Curtailment Absorption and Peak Reduction in Sri Lanka.

**WARNING:** This notebook uses illustrative, synthetic EV arrivals/energy/dwell durations. These are **not calibrated to Sri Lankan public chargers** and do not establish renewable curtailment or peak reduction. It builds the engineering baseline against which a future optimized schedule will be compared.

No original PUCSL/CEB data are modified; diagnostic outputs go under `results/ev_baseline/`.


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "ev_charging_baseline.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "scripts" / "ev_charging_baseline.py").is_file():
    raise FileNotFoundError("Open this notebook from the grid-pulse-ev-starter project (or its notebooks folder)")
sys.path.insert(0, str(ROOT / "scripts"))
from ev_charging_baseline import generate_synthetic_sessions, simulate_uncontrolled, run_experiment, TZ
print("Project root:", ROOT)


## 1. Reproducible experiment settings

Try different seeds and session counts later as **sensitivity scenarios**, not claims about Sri Lanka. A seven-day period in April 2024 was chosen to allow an optional alignment with your existing 2024 grid dataset.

In [ ]:
START_DATE = "2024-04-01"
DAYS = 7
SESSIONS_PER_DAY = 20
RANDOM_SEED = 42
OUTDIR = ROOT / "results" / "ev_baseline"
metrics = run_experiment(OUTDIR, START_DATE, DAYS, SESSIONS_PER_DAY, RANDOM_SEED)
print(json.dumps(metrics, indent=2))
print("Saved to:", OUTDIR)


## 2. Check the generated sessions and delivered energy

`energy_request_kwh` is an assumed target, not ACN-Data `kWhDelivered` (which measures historical delivered electricity). A driver may leave with unmet demand.

In [ ]:
sessions = pd.read_csv(OUTDIR / "synthetic_ev_sessions.csv")
outcomes = pd.read_csv(OUTDIR / "uncontrolled_session_outcomes.csv")
profile = pd.read_csv(OUTDIR / "uncontrolled_site_load_15min.csv")
display(sessions.head(8))
display(outcomes.head(8))
print("Energy conservation check (kWh):", round(profile.site_charging_kw.sum() * 0.25 - outcomes.delivered_kwh.sum(), 8))
assert abs(profile.site_charging_kw.sum() * 0.25 - outcomes.delivered_kwh.sum()) < 1e-5


## 3. Uncontrolled site charging power

Immediate charging at each plugged-in vehicle's power ceiling. This baseline assumes sufficient EVSEs and does **not** impose a shared site capacity; therefore it is not yet a site-feasible charging schedule.

In [ ]:
profile["t"] = pd.to_datetime(profile.interval_start_local, utc=True).dt.tz_convert(TZ)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(profile.t, profile.site_charging_kw)
ax.set_title("Illustrative uncontrolled EV charging: aggregate load")
ax.set_ylabel("Incremental site charging demand (kW)")
ax.set_xlabel("Sri Lanka local time")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()


## 4. Session feasibility and unmet energy

Energy requests exceeding `max_power_kw × parking hours` may be physically infeasible even with immediate charging. An optimizer cannot satisfy those requests without changing the assumptions.

In [ ]:
print("Fully served:", int(outcomes.fully_served.sum()), "of", len(outcomes))
print("Unmet energy (kWh):", round(outcomes.unmet_kwh.sum(), 2))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(outcomes.unmet_kwh, bins=12)
ax.set_title("Illustrative unmet energy per synthetic session")
ax.set_xlabel("Unmet request (kWh)")
ax.set_ylabel("Sessions")
plt.tight_layout(); plt.show()


## 5. Optional: overlay incremental EV load on the 2024 PUCSL hourly dispatch **proxy**

Only attempt this where `hour_status == complete` and `is_system_event_day == False`. This is illustrative **additional** EV charging on top of PUCSL reported dispatch (which may already include some EV consumption). It is not verified national demand, observed charging growth, or an optimization result. Missing/flagged grid hours stay missing.

In [ ]:
hourly_file = ROOT / "data" / "processed" / "grid" / "pucsl_2024_hourly_net_demand.csv"
if not hourly_file.exists():
    print("Optional grid overlay skipped: local hourly proxy file does not exist.")
else:
    grid = pd.read_csv(hourly_file)
    grid["t"] = pd.to_datetime(grid.hour_start_local, utc=True).dt.tz_convert(TZ)
    site = profile.set_index("t").site_charging_kw.resample("h").mean().rename("extra_ev_mw") / 1000.0
    grid = grid.merge(site, left_on="t", right_index=True, how="inner")
    is_event = grid.is_system_event_day.astype(str).str.lower().eq("true")
    eligible = grid.hour_status.eq("complete") & ~is_event
    graph = grid.loc[eligible].copy()
    graph["dispatch_plus_extra_ev_mw"] = graph.total_dispatch_reported_mw + graph.extra_ev_mw
    print("Quality-eligible hours displayed:", len(graph), "of", len(grid))
    if graph.empty:
        print("No eligible hours in selected period; no values were fabricated.")
    else:
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(graph.t, graph.total_dispatch_reported_mw, label="PUCSL reported dispatch (proxy)")
        ax.plot(graph.t, graph.dispatch_plus_extra_ev_mw, label="Proxy + illustrative additional EV")
        ax.set_title("Illustrative incremental EV demand on a PUCSL dispatch proxy")
        ax.set_ylabel("MW, not verified national demand")
        ax.legend()
        fig.autofmt_xdate(); plt.tight_layout(); plt.show()


## 6. Interpretation and next research task

**This stage proves a reproducible baseline implementation, not grid effectiveness.**

Next: obtain empirical EV-session records with provenance and access terms; test and document fields, especially requested versus delivered energy; add session-generator calibration and transferability scenarios. Then add charger/site constraints and compare optimized schedules against this uncontrolled baseline.

Official candidate source: [ACN-Data](https://ev.caltech.edu/dataset) (California campus/workplace; not representative of Sri Lankan public charging without further validation).